# 04 用 PyTorch 拟合功率模型

目标：用 PyTorch 写一个最小训练循环，看清楚模型训练做了什么。

我们只用一个线性层预测 `log10(Power kW)`。这不是为了炫技，而是为了理解：特征、loss、优化器、验证误差。

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch

BASE = Path.cwd()
if BASE.name == 'notebooks':
    BASE = BASE.parent
DATA = BASE.parent / '2026-09-17 第一次课' / '重写论文' / 'analysis' / 'inputs' / 'top500_full.csv'
top500 = pd.read_csv(DATA)
reported = top500[top500['Power (kW)'].notna()].copy()
reported['has_accelerator'] = reported['Accelerator/Co-Processor'].notna().astype(float)
reported = reported.sample(frac=1, random_state=42).reset_index(drop=True)

X_np = np.column_stack([
    np.log10(reported['Rmax [TFlop/s]'].to_numpy()),
    reported['has_accelerator'].to_numpy(),
    (reported['Year'].to_numpy() - 2020) / 10,
]).astype('float32')
y_np = np.log10(reported['Power (kW)'].to_numpy()).astype('float32').reshape(-1, 1)

# 标准化特征，训练更稳定。
X_mean = X_np.mean(axis=0, keepdims=True)
X_std = X_np.std(axis=0, keepdims=True)
X_np = (X_np - X_mean) / X_std

split = int(len(X_np) * 0.8)
X_train = torch.tensor(X_np[:split])
y_train = torch.tensor(y_np[:split])
X_val = torch.tensor(X_np[split:])
y_val = torch.tensor(y_np[split:])

print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)

In [ ]:
torch.manual_seed(0)
model = torch.nn.Linear(3, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = torch.nn.MSELoss()

history = []
for epoch in range(401):
    model.train()
    optimizer.zero_grad()
    train_pred = model(X_train)
    train_loss = loss_fn(train_pred, y_train)
    train_loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = loss_fn(val_pred, y_val)
        history.append((epoch, float(train_loss), float(val_loss)))
        print(f'epoch {epoch:03d} | train loss {train_loss.item():.4f} | val loss {val_loss.item():.4f}')

In [ ]:
with torch.no_grad():
    pred_log = model(X_val).numpy().reshape(-1)
true_log = y_val.numpy().reshape(-1)
rmse = np.mean((pred_log - true_log) ** 2) ** 0.5
median_mult_error = np.median(10 ** np.abs(pred_log - true_log))
print(f'Validation log10 RMSE: {rmse:.3f}')
print(f'Median multiplicative error: {median_mult_error:.2f}x')

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4.5))
plt.scatter(10 ** true_log, 10 ** pred_log, alpha=0.75)
lo = min((10 ** true_log).min(), (10 ** pred_log).min())
hi = max((10 ** true_log).max(), (10 ** pred_log).max())
plt.plot([lo, hi], [lo, hi], '--', color='gray')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Observed power (kW)')
plt.ylabel('PyTorch prediction (kW)')
plt.title('Simple PyTorch linear model')
plt.show()

课堂讨论：

1. 这个 PyTorch 模型和 `sklearn.Ridge` 在思想上有什么相同？
2. 为什么验证误差不能自动支持全球外推？
3. 如果想写进论文，哪些训练细节必须说明？